In [2]:
# Core imports
import os
import json
import requests
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ Setup complete")

✅ Setup complete


In [4]:
import ollama

# Test simple prompt
response = ollama.chat(
    model="llama3.2",
    messages=[
        {"role": "user", "content": "Say hello in one sentence"}
    ]
)

print(response["message"]["content"])

Hello.


In [7]:
SYSTEM_PROMPT = """
You are PersonaForge, an intelligent AI assistant designed for professional and insightful conversations.

Rules:
- Be clear, concise, and structured
- Provide thoughtful, high-quality responses
- Avoid unnecessary verbosity
- If unsure, say you don't know
- Maintain a professional tone
"""

In [8]:
def chat_with_llama(user_input, chat_history=[]):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ] + chat_history + [
        {"role": "user", "content": user_input}
    ]

    response = ollama.chat(
        model="llama3.2",
        messages=messages
    )

    reply = response["message"]["content"]

    # Update history
    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": reply})

    return reply, chat_history

In [9]:
chat_history = []

reply, chat_history = chat_with_llama("Explain AI in 2 lines", chat_history)
print(reply)

Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks typically requiring human intelligence, such as learning, problem-solving, and decision-making. By mimicking human cognition, AI enables machines to analyze complex data, recognize patterns, and generate insights with increasing accuracy and efficiency.


In [10]:
GENERATOR_MODEL = "llama3.2"
EVALUATOR_MODEL = "mistral"

In [11]:
def generator_agent(user_input, chat_history):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ] + chat_history + [
        {"role": "user", "content": user_input}
    ]

    response = ollama.chat(
        model=GENERATOR_MODEL,
        messages=messages
    )

    return response["message"]["content"]

In [12]:
EVALUATOR_PROMPT = """
You are an AI evaluator.

Your job is to evaluate the assistant's response based on:
- clarity
- correctness
- completeness
- professionalism

Return output in JSON format:
{
  "score": number (1-10),
  "verdict": "approve" or "improve",
  "feedback": "short feedback",
  "improved_response": "only if improvement needed"
}
"""

In [13]:
def evaluator_agent(user_input, response):
    messages = [
        {"role": "system", "content": EVALUATOR_PROMPT},
        {"role": "user", "content": f"""
User Question:
{user_input}

Assistant Response:
{response}
"""}
    ]

    eval_response = ollama.chat(
        model=EVALUATOR_MODEL,
        messages=messages
    )

    return eval_response["message"]["content"]

In [16]:
import json

def multi_agent_chat(user_input, chat_history=[]):
    # Step 1: Generate response
    raw_response = generator_agent(user_input, chat_history)

    # Step 2: Evaluate response
    evaluation = evaluator_agent(user_input, raw_response)

    try:
        eval_json = json.loads(evaluation)
    except:
        # fallback if JSON fails
        return raw_response, chat_history

    # Step 3: Decision
    if eval_json["verdict"] == "approve":
        final_response = raw_response
    else:
        final_response = eval_json.get("improved_response", raw_response)

    # Update history
    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": final_response})

    return final_response, chat_history

In [ ]:
chat_history = []

reply, chat_history = multi_agent_chat(
    "Explain AI like I'm a beginner but keep it professional",
    chat_history
)

print(reply)